<a href="https://colab.research.google.com/github/lubabasadiyanp/sentimental-analysis-of-product-review/blob/sebin/Data%20Understanding%20%26%20preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step1:Load Dataset & preprocessing






In [1]:
import pandas as pd
from datasets import load_dataset

# Yelp
dataset = load_dataset('yelp_review_full')
df = dataset['train'].to_pandas()
df = df[['text', 'label']].sample(n=10000, random_state=42)
df['stars'] = df['label'] + 1

# Sentiment mapping
def rating_to_sentiment(r):
    if r >= 4:   return 'Positive'
    elif r == 3: return 'Neutral'
    else:        return 'Negative'

df['sentiment'] = df['stars'].apply(rating_to_sentiment)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

yelp_review_full/train-00000-of-00001.pa(…):   0%|          | 0.00/299M [00:00<?, ?B/s]

yelp_review_full/test-00000-of-00001.par(…):   0%|          | 0.00/23.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/650000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [2]:
pd.set_option('display.max_columns',None)
df.head(5)

,text,label,stars,sentiment
177288,"First of all i'm not a big fan of buffet, i tr...",0,1,Negative
238756,Thanks Yelp. I was looking for the words to de...,1,2,Negative
604225,Service was so-so. They were receiving a deliv...,2,3,Neutral
2838,Stamoolis Brothers is one of the Strip Distric...,2,3,Neutral
586957,I want to give a 2 stars because the service s...,0,1,Negative


In [3]:
df.info

<bound method DataFrame.info of                                                      text  label  stars  \
177288  First of all i'm not a big fan of buffet, i tr...      0      1   
238756  Thanks Yelp. I was looking for the words to de...      1      2   
604225  Service was so-so. They were receiving a deliv...      2      3   
2838    Stamoolis Brothers is one of the Strip Distric...      2      3   
586957  I want to give a 2 stars because the service s...      0      1   
...                                                   ...    ...    ...   
456268  Four Hours of my life I'd like to have back!\n...      0      1   
259135  Small stadium and would have great views all a...      1      2   
496284  I think Porktropolis has the BBQ thing nailed....      2      3   
512684  Best CU ever!!! Great customer service!!! Know...      4      5   
597788  Came in for late lunch on a Tuesday. I sat at ...      2      3   

       sentiment  
177288  Negative  
238756  Negative  
604225   Neutral  
2838     Neutral  
586957  Negative  
...          ...  
456268  Negative  
259135  Negative  
496284   Neutral  
512684  Positive  
597788   Neutral  

[10000 rows x 4 columns]>

In [4]:
df.describe()

,label,stars
count,10000.000000,10000.000000
mean,1.998100,2.998100
std,1.410849,1.410849
min,0.000000,1.000000
25%,1.000000,2.000000
50%,2.000000,3.000000
75%,3.000000,4.000000
max,4.000000,5.000000


In [5]:
df.columns

Index(['text', 'label', 'stars', 'sentiment'], dtype='object')

In [6]:
df.isnull().sum()

,0
text,0
label,0
stars,0
sentiment,0


In [7]:
df.duplicated().sum()

np.int64(0)

# Step 2:Text Cleaning

In [8]:
import re

def clean_text(text):
    text = str(text).lower().strip()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^\w\s!?.,\'"-]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['text'].apply(clean_text)
df.to_csv('preprocessed_data.csv', index=False)

# Step 3: SVM + TF-IDF

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['sentiment'],
    test_size=0.2, random_state=42, stratify=df['sentiment']
)

tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

svm = SVC(kernel='linear', C=1.0)
svm.fit(X_train_tfidf, y_train)

preds = svm.predict(X_test_tfidf)
print(classification_report(y_test, preds))

              precision    recall  f1-score   support

    Negative       0.78      0.86      0.82       805
     Neutral       0.53      0.37      0.44       396
    Positive       0.80      0.84      0.82       799

    accuracy                           0.75      2000
   macro avg       0.70      0.69      0.69      2000
weighted avg       0.74      0.75      0.74      2000



# Step 4:results for comparison

In [10]:
import pickle
pickle.dump(tfidf, open('tfidf_vectorizer.pkl', 'wb'))
pickle.dump(svm,   open('svm_model.pkl', 'wb'))

# Save metrics as dict for Person 3 to use in comparison
svm_report = classification_report(y_test, preds, output_dict=True)

In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu
